# 01b · Catálogo propio → esquema común

Pasa las 500 prendas del catálogo propio al mismo esquema que se usará con Polyvore (revisión §12.12). El catálogo **no** aporta etiquetas de compatibilidad: solo prendas para los clósets (70 % Polyvore / 30 % propias).

**Entradas:** `data/raw/prendas_base_500_mejorado.csv` y, si existe, la planilla de revisión manual `data/revision/catalogo_propio_revision.xlsx`.
**Salidas:** `data/processed/hito3/catalogo_propio.csv` y `results/hito3/catalogo_propio_resumen.csv`.

Las reglas de mapeo están en `src/modelo_fly/catalogo_propio.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.catalogo_propio import aplicar_revision, transformar
from modelo_fly.esquema import COLUMNAS

RAW = ROOT / "data" / "raw" / "prendas_base_500_mejorado.csv"
REVISION = ROOT / "data" / "revision" / "catalogo_propio_revision.xlsx"
SALIDA = ROOT / "data" / "processed" / "hito3" / "catalogo_propio.csv"
RESUMEN = ROOT / "results" / "hito3" / "catalogo_propio_resumen.csv"
for ruta in (REVISION, SALIDA, RESUMEN):
    ruta.parent.mkdir(parents=True, exist_ok=True)

crudo = pd.read_csv(RAW, sep=";", encoding="utf-8")
print(crudo.shape)

(500, 9)


## 1. Problemas del catálogo crudo

Antes de mapear se miden las contradicciones internas. El color, la categoría y la época de `descripcion` coinciden siempre con sus columnas; el estilo no.

In [2]:
estilo_desc = crudo["descripcion"].str.rsplit("Estilo ", n=1).str[-1]
chequeos = pd.Series({
    "color = descripción": sum(d.startswith(c + " ") for c, d in zip(crudo["color"], crudo["descripcion"])),
    "época = descripción": sum(f" {e} - " in d for e, d in zip(crudo["epoca"], crudo["descripcion"])),
    "estilo = descripción": int((estilo_desc == crudo["estilo"]).sum()),
}, name="prendas (de 500)")
display(chequeos.to_frame())

print("Niveles por columna en el crudo:")
display(crudo[["color", "patron", "estilo", "epoca", "tamaño"]].nunique().to_frame("niveles"))
print("Talla por categoría (las tallas parecen asignadas al azar):")
display(pd.crosstab(crudo["categoria"], crudo["tamaño"]))

,prendas (de 500)
color = descripción,500
época = descripción,500
estilo = descripción,66


Niveles por columna en el crudo:


,niveles
color,69
patron,8
estilo,8
epoca,8
tamaño,13


Talla por categoría (las tallas parecen asignadas al azar):


tamaño,36,37,38,39,40,41,42,L,M,OneSize,S,XL,XS
categoria,,,,,,,,,,,,,
Blusas,5,1,2,4,3,3,4,4,5,5,4,7,6
Calzado,3,2,0,3,0,1,2,7,2,3,4,1,5
Camiseta,4,11,8,4,6,5,12,7,7,6,5,5,8
Chaqueta,6,5,4,6,6,5,2,3,4,7,4,7,7
Faldas y minifaldas,5,6,4,5,6,3,1,4,7,3,5,10,7
Pantalon,1,9,12,1,6,10,5,6,4,7,5,5,4
Short,3,2,1,1,6,2,5,1,3,5,7,3,5
Vestido,2,3,9,6,6,5,9,7,4,6,4,7,7


## 2. Mapeo al esquema común

| Columna cruda | Pasa a | Regla |
|---|---|---|
| `categoria` | `slot` | Camiseta/Blusa → arriba; Pantalón/Falda/Short → abajo; Vestido; Chaqueta → exterior; Calzado |
| `color` (69) | `tono` (15) + `familia_color` | Sinónimos unificados; denim = neutro; "Vintage", "Desteñido", "Pastel"… → desconocido |
| `patron` | `patron` + `material` | Cuero, Denim, Encaje, Terciopelo son material (patrón liso) |
| `estilo` + descripción | `estilo` + `calce` | Se reparten; si chocan, **manda la columna** y `conflicto_estilo = 1` |
| `epoca` | `epoca` | Décadas → 1950s…1990s; Tumblr → 2010s; Retro/Vintage → vintage_generico |
| `categoria` | `largo` | Short → corto; Pantalón → largo; resto desconocido |
| `tamaño` | `talla` | Ropa: 36 XS, 37 S, 38 M, 39 L, 40 XL, 41 XXL, 42 XXXL (tallas peruanas). Calzado: sin cambio |
| `descripcion` | `texto` | Texto plano: minúsculas, sin tildes ni símbolos |

In [3]:
catalogo = transformar(crudo)

ejemplos = ["P0001", "P0002", "P0003", "P0006"]
display(catalogo.loc[catalogo["item_id"].isin(ejemplos),
                     ["item_id", "estilo_original", "estilo_descripcion", "estilo", "calce", "conflicto_estilo"]])

reparto = pd.Series({
    "coinciden": int((catalogo["estilo_original"] == catalogo["estilo_descripcion"]).sum()),
    "se complementan (estilo + calce)": int(((catalogo["estilo_original"] != catalogo["estilo_descripcion"])
                                             & (catalogo["conflicto_estilo"] == 0)).sum()),
    "contradicción (manda la columna)": int(catalogo["conflicto_estilo"].sum()),
}, name="prendas")
display(reparto.to_frame())

,item_id,estilo_original,estilo_descripcion,estilo,calce,conflicto_estilo
0,P0001,Vintage,Oversize,vintage,oversize,0
1,P0002,Formal,Recto,formal,recto,0
2,P0003,Vintage,Vintage,vintage,desconocido,0
5,P0006,Oversize,Recto,desconocido,oversize,1


,prendas
coinciden,66
se complementan (estilo + calce),229
contradicción (manda la columna),205


In [4]:
print("Tallas de ropa (sin calzado), original → nueva:")
display(pd.crosstab(catalogo.loc[catalogo["slot"] != "calzado", "talla_original"],
                    catalogo.loc[catalogo["slot"] != "calzado", "talla"]))
display(catalogo[["item_id", "texto_original", "texto"]].head(5))

Tallas de ropa (sin calzado), original → nueva:


talla,L,M,OneSize,S,XL,XS,XXL,XXXL
talla_original,,,,,,,,
36,0,0,0,0,0,26,0,0
37,0,0,0,37,0,0,0,0
38,0,40,0,0,0,0,0,0
39,27,0,0,0,0,0,0,0
40,0,0,0,0,39,0,0,0
41,0,0,0,0,0,0,33,0
42,0,0,0,0,0,0,0,38
L,32,0,0,0,0,0,0,0
M,0,34,0,0,0,0,0,0


,item_id,texto_original,texto
0,P0001,Azul Cielo camiseta Años 50 - Estilo Oversize,azul cielo camiseta anos 50 estilo oversize
1,P0002,Blanco camiseta Años 50 - Estilo Recto,blanco camiseta anos 50 estilo recto
2,P0003,Amarillo Mantequilla camiseta Años 50 - Estilo...,amarillo mantequilla camiseta anos 50 estilo v...
3,P0004,Amarillo Mantequilla camiseta Años 50 - Estilo...,amarillo mantequilla camiseta anos 50 estilo o...
4,P0005,Rosa Pastel camiseta Años 50 - Estilo Ajustado,rosa pastel camiseta anos 50 estilo ajustado


In [5]:
for col in COLUMNAS:
    print(col, catalogo[col].value_counts().to_dict())

slot {'abajo': 185, 'arriba': 141, 'vestido': 75, 'exterior': 66, 'calzado': 33}
tono {'negro': 87, 'gris': 56, 'blanco': 45, 'beige': 44, 'azul': 41, 'teal': 31, 'marron': 31, 'crema': 26, 'rosa': 21, 'naranja': 20, 'desconocido': 20, 'verde': 17, 'amarillo': 16, 'plateado': 16, 'rojo': 16, 'morado': 13}
familia_color {'neutro': 297, 'fuerte': 114, 'suave': 71, 'desconocido': 18}
patron {'liso': 312, 'holografico': 67, 'floral': 63, 'grafico': 58}
material {'desconocido': 260, 'encaje': 63, 'terciopelo': 62, 'denim': 58, 'cuero': 57}
estilo {'vintage': 90, 'formal': 88, 'deportivo': 86, 'casual': 83, 'athleisure': 83, 'desconocido': 70}
calce {'desconocido': 193, 'oversize': 94, 'recto': 92, 'ajustado': 88, 'no_aplica': 33}
epoca {'vintage_generico': 135, '2010s': 86, '1980s': 69, '1990s': 69, '1950s': 47, '1960s': 47, '1970s': 47}
largo {'desconocido': 348, 'largo': 75, 'corto': 44, 'no_aplica': 33}


## 3. Revisión manual

Las prendas con señales poco creíbles quedan como `revisar` (no se excluyen solas). La planilla `data/revision/catalogo_propio_revision.xlsx` tiene las 500 prendas, primero las que hay que revisar.

**Cómo llenarla:** en `decision` poner `ok`, `corregido` o `excluido`; si es `corregido`, escribir el valor nuevo en la columna `corr_<atributo>` (con el vocabulario de `src/modelo_fly/esquema.py`); `nota` es libre. Luego volver a ejecutar este notebook. Las decisiones ya escritas no se borran al regenerar la planilla.

**Regla de exclusión (escrita antes de revisar):** se excluye una prenda solo si la combinación no existe en tiendas reales o si un atributo es ilegible y no se puede corregir desde el texto.

In [6]:
print(catalogo["calidad"].value_counts().to_dict())
motivos = catalogo["motivo_revision"].str.split("; ").explode()
display(motivos[motivos != ""].value_counts().to_frame("prendas"))

COLS_REVISION = ["item_id", "categoria", "texto_original", "color_original", "patron_original", "talla", "talla_original",
                 *COLUMNAS, "conflicto_estilo", "motivo_revision", "calidad"]
COLS_MANUALES = ["decision", "nota", *[f"corr_{c}" for c in COLUMNAS]]

planilla = catalogo[COLS_REVISION].copy()
if REVISION.exists():
    previa = pd.read_excel(REVISION, dtype=str)
    manual = previa[["item_id", *[c for c in COLS_MANUALES if c in previa.columns]]]
    planilla = planilla.merge(manual, on="item_id", how="left")
    catalogo = aplicar_revision(catalogo, previa)
    print("Revisión aplicada:", int(previa["decision"].notna().sum()), "decisiones")
for c in COLS_MANUALES:
    if c not in planilla.columns:
        planilla[c] = pd.NA

planilla = planilla.sort_values(["calidad", "item_id"], key=lambda s: s.ne("revisar") if s.name == "calidad" else s)
planilla.to_excel(REVISION, index=False, sheet_name="revision")
print("Planilla:", REVISION.relative_to(ROOT), "| filas", len(planilla))

{'ok': 347, 'revisar': 153}


,prendas
motivo_revision,
color no es un color,20
camiseta de cuero,15
athleisure de cuero,14
athleisure de terciopelo,14
camiseta de denim,13
formal con patrón grafico,12
formal con patrón holografico,12
deportivo de terciopelo,11
camiseta de terciopelo,10


Revisión aplicada: 153 decisiones


Planilla: data\revision\catalogo_propio_revision.xlsx | filas 500


## 4. Salidas

In [7]:
# utf-8-sig: con BOM, para que Excel muestre bien las tildes de las columnas *_original
catalogo.to_csv(SALIDA, index=False, encoding="utf-8-sig")

resumen = pd.concat([
    catalogo[col].value_counts().rename_axis("valor").reset_index(name="prendas").assign(columna=col)
    for col in [*COLUMNAS, "calidad", "conflicto_estilo"]
])[["columna", "valor", "prendas"]]
resumen.to_csv(RESUMEN, index=False, encoding="utf-8-sig")

desconocidos = (catalogo[COLUMNAS] == "desconocido").mean().mul(100).round(1)
display(desconocidos.to_frame("% desconocido"))
print("Guardado:", SALIDA.relative_to(ROOT), catalogo.shape)
print("Guardado:", RESUMEN.relative_to(ROOT))
print("Utilizables (no excluidas):", int(catalogo["calidad"].ne("excluido").sum()), "de", len(catalogo))

,% desconocido
slot,0.0
tono,4.0
familia_color,3.6
patron,0.0
material,52.0
estilo,14.0
calce,38.6
epoca,0.0
largo,69.6


Guardado: data\processed\hito3\catalogo_propio.csv (500, 24)
Guardado: results\hito3\catalogo_propio_resumen.csv
Utilizables (no excluidas): 500 de 500
